# Assignment 8: Noise Generation from PDFs


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform, norm, gamma, expon, rayleigh

gray_img = cv2.imread('Sunflower_from_Silesia2.jpg', cv2.IMREAD_GRAYSCALE)
h, w = gray_img.shape


## Noise Generation


In [ ]:
np.random.seed(42)

U1 = np.random.uniform(1e-7, 1.0 - 1e-7, (h, w))
U2 = np.random.uniform(1e-7, 1.0 - 1e-7, (h, w))

a_unif, b_unif = -30.0, 30.0
noise_uniform = a_unif + (b_unif - a_unif) * U1

mu_gauss, sigma_gauss = 0.0, 25.0
noise_gaussian = mu_gauss + sigma_gauss * np.sqrt(-2.0 * np.log(U1)) * np.cos(2.0 * np.pi * U2)

a_erlang, b_erlang = 0.2, 3
U_erlang = np.random.uniform(1e-7, 1.0, (b_erlang, h, w))
noise_erlang = - (1.0 / a_erlang) * np.sum(np.log(U_erlang), axis=0)

a_exp = 0.08
noise_exp = - (1.0 / a_exp) * np.log(1.0 - U1)

a_ray, b_ray = 0.0, 800.0
noise_rayleigh = a_ray + np.sqrt(-b_ray * np.log(1.0 - U1))


## Histograms of Generated Noise vs Theoretical Distributions


In [ ]:
plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
plt.hist(noise_uniform.flatten(), bins=60, density=True, color='skyblue', edgecolor='black', alpha=0.7, label='Derived')
x_vals = np.linspace(a_unif - 10, b_unif + 10, 200)
plt.plot(x_vals, uniform.pdf(x_vals, loc=a_unif, scale=b_unif - a_unif), 'r-', lw=2, label='Theoretical PDF')
plt.title('1. Uniform Noise')
plt.xlabel('Noise Value')
plt.ylabel('Density')
plt.legend()

plt.subplot(2, 3, 2)
plt.hist(noise_gaussian.flatten(), bins=60, density=True, color='salmon', edgecolor='black', alpha=0.7, label='Derived (Box-Muller)')
x_vals = np.linspace(-4 * sigma_gauss, 4 * sigma_gauss, 200)
plt.plot(x_vals, norm.pdf(x_vals, loc=mu_gauss, scale=sigma_gauss), 'r-', lw=2, label='Theoretical PDF')
plt.title('2. Gaussian Noise')
plt.xlabel('Noise Value')
plt.legend()

plt.subplot(2, 3, 3)
plt.hist(noise_erlang.flatten(), bins=60, density=True, color='mediumseagreen', edgecolor='black', alpha=0.7, label='Derived Erlang')
x_vals = np.linspace(0, np.percentile(noise_erlang, 99.5), 200)
plt.plot(x_vals, gamma.pdf(x_vals, a=b_erlang, scale=1.0/a_erlang), 'r-', lw=2, label='Theoretical PDF')
plt.title('3. Erlang Noise')
plt.xlabel('Noise Value')
plt.legend()

plt.subplot(2, 3, 4)
plt.hist(noise_exp.flatten(), bins=60, density=True, color='gold', edgecolor='black', alpha=0.7, label='Derived Exp')
x_vals = np.linspace(0, np.percentile(noise_exp, 99), 200)
plt.plot(x_vals, expon.pdf(x_vals, scale=1.0/a_exp), 'r-', lw=2, label='Theoretical PDF')
plt.title('4. Exponential Noise')
plt.xlabel('Noise Value')
plt.legend()

plt.subplot(2, 3, 5)
plt.hist(noise_rayleigh.flatten(), bins=60, density=True, color='plum', edgecolor='black', alpha=0.7, label='Derived Rayleigh')
x_vals = np.linspace(0, np.percentile(noise_rayleigh, 99.5), 200)
scale_r = np.sqrt(b_ray / 2.0)
plt.plot(x_vals, rayleigh.pdf(x_vals, loc=a_ray, scale=scale_r), 'r-', lw=2, label='Theoretical PDF')
plt.title('5. Rayleigh Noise')
plt.xlabel('Noise Value')
plt.legend()

plt.tight_layout()
plt.show()


## Noise Corrupted Images


In [ ]:
def add_noise_to_image(image, noise):
    corrupted = image.astype(np.float64) + noise
    return np.uint8(np.clip(corrupted, 0, 255))

img_unif = add_noise_to_image(gray_img, noise_uniform)
img_gauss = add_noise_to_image(gray_img, noise_gaussian)
img_erlang = add_noise_to_image(gray_img, noise_erlang - np.mean(noise_erlang))
img_exp = add_noise_to_image(gray_img, noise_exp - np.mean(noise_exp))
img_ray = add_noise_to_image(gray_img, noise_rayleigh - np.mean(noise_rayleigh))

plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.title('Original Clean Image')
plt.imshow(gray_img, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.title('Image + Uniform Noise')
plt.imshow(img_unif, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.title('Image + Gaussian Noise')
plt.imshow(img_gauss, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.title('Image + Erlang Noise')
plt.imshow(img_erlang, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 5)
plt.title('Image + Exponential Noise')
plt.imshow(img_exp, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 6)
plt.title('Image + Rayleigh Noise')
plt.imshow(img_ray, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()
